# Linear Regression

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 20)

## Review: Watch Out For

Last class we measured linear association with $r$. Four things to watch out for: false conclusions of causation, nonlinearity, outliers, and ecological correlations. First, a quick look back at the hybrid cars.

In [ ]:
hybrid <- read_csv("data/hybrid.csv", show_col_types = FALSE)

ggplot(hybrid, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$acceleration, hybrid$msrp)

### Switching Axes

What happens to $r$ if we put `msrp` on the x-axis instead?

In [ ]:
ggplot(hybrid, aes(x = msrp, y = acceleration)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$msrp, hybrid$acceleration)

### Nonlinearity

Each `y` is `x` squared, so knowing `x` tells us `y` exactly. What is $r$?

In [ ]:
nonlinear <- tibble(
  x = seq(-4, 4, by = 0.5),
  y = x^2
)

ggplot(nonlinear, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(nonlinear$x, nonlinear$y)

### Outliers

Four points on a line, then the same points plus one more.

In [ ]:
line <- tibble(x = 1:4, y = 1:4)

ggplot(line, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(line$x, line$y)

In [ ]:
outlier <- tibble(x = 1:5, y = c(1, 2, 3, 4, 0))

ggplot(outlier, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(outlier$x, outlier$y)

### Ecological Correlations

`sat2014.csv` has one row for each state and DC, with the state's average 2014 SAT scores and its participation rate: the percent of students who took the SAT.

In [ ]:
sat2014 <- read_csv("data/sat2014.csv", show_col_types = FALSE) |>
  arrange(State)

sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(sat2014$`Critical Reading`, sat2014$Math)

Each point is a state, not a student. Why are the state averages so closely related? Look at who took the test. We code the participation rate as low (25% or less), medium (more than 25% and up to 75%), or high (more than 75%).

In [ ]:
sat2014 <- sat2014 |>
  mutate(rate_code = case_when(
    `Participation Rate` <= 25 ~ "low",
    `Participation Rate` <= 75 ~ "medium",
    TRUE ~ "high"
  ))

sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math, color = rate_code)) +
  geom_point(size = 2) +
  scale_color_manual(values = c(low = "goldenrod", medium = "dodgerblue3", high = "red"),
                     breaks = c("low", "medium", "high")) +
  theme_minimal(base_size = 16)

In [ ]:
sat2014 |>
  filter(rate_code == "low")

Where few students took the SAT, the ones who did often had a reason to, like applying to a private or out-of-state college. Those states have the highest averages. The correlation of 0.98 describes state averages, not individual students.

## Part 1: Predicting Heights

`family_heights.csv` has one row per adult child, from data collected by Francis Galton in the 1880s. Heights are in inches. The midparent height is an average of the two parents' heights. (Galton multiplied each mother's height by 1.08 first, to put mothers and fathers on the same scale.)

In [ ]:
original <- read_csv("data/family_heights.csv", show_col_types = FALSE)

heights <- tibble(
  MidParent = original$midparentHeight,
  Child = original$childHeight
)

heights

In [ ]:
heights |>
  ggplot(aes(x = MidParent, y = Child)) +
  geom_point(alpha = 0.5) +
  labs(title = "Child's height and midparent height",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

### Nearest Neighbor Regression

To predict the height of a child whose midparent height is `mpht`, average the heights of the children whose midparent height is within half an inch of `mpht`.

In [ ]:
predict_child <- function(mpht) {
  close_points <- heights |>
    filter(MidParent >= mpht - 0.5, MidParent < mpht + 0.5)
  mean(close_points$Child)
}

predict_child(70.48)

`predict_child()` works on one height at a time, so we use `map_dbl()` to run it once for each midparent height.

In [ ]:
heights_with_predictions <- heights |>
  mutate(Prediction = map_dbl(MidParent, predict_child))

heights_with_predictions

The gold points are the predictions: the **graph of averages**.

In [ ]:
heights_with_predictions |>
  ggplot(aes(x = MidParent)) +
  geom_point(aes(y = Child), alpha = 0.5) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2.5) +
  labs(title = "Nearest neighbor predictions (gold)",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

### What About 74.5 Inches?

New parents have a midparent height of 74.5 inches. What does nearest neighbor predict?

In [ ]:
predict_child(74.5)

`NaN` means "not a number." No midparent heights in the data are within half an inch of 74.5, so there's nothing to average. Here are the tallest parents:

In [ ]:
heights |>
  filter(MidParent > 73) |>
  count(MidParent)

## Part 2: Where Is the Prediction Line?

`r_table()` makes 1,000 made-up points in standard units with a correlation close to `r`. You don't have to understand how it works. The `set.seed()` inside makes the same points every time, so these match the slides.

In [ ]:
r_table <- function(r) {
  set.seed(42)
  x <- rnorm(1000)
  z <- rnorm(1000)
  tibble(x = x, y = r * x + sqrt(1 - r^2) * z)
}

`su_plot()` draws a scatter plot in standard units with the same axes every time. We'll add lines and points to it with `+`.

In [ ]:
su_plot <- function(table) {
  table |>
    ggplot(aes(x = x, y = y)) +
    geom_point(alpha = 0.5) +
    coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
    labs(x = "x (standard units)", y = "y (standard units)") +
    theme_minimal(base_size = 16)
}

`nn_prediction_example()` is nearest neighbor regression for these points. It averages `y` for the `x` values within 0.25 of `x_val`, using the table named `example`.

In [ ]:
nn_prediction_example <- function(x_val) {
  neighbors <- example |>
    filter(between(x, x_val - 0.25, x_val + 0.25))
  mean(neighbors$y)
}

### r = 0.99

In [ ]:
example <- r_table(0.99)
su_plot(example)

In [ ]:
nn_prediction_example(-2.25)

Predict `y` for every point, then plot the predictions (gold) with the line y = x (red).

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

### r = 0

In [ ]:
example <- r_table(0)
su_plot(example)

Where should the prediction line be? The red line y = x would predict a high `y` for a high `x`.

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

The averages stay near 0, so the flat line y = 0 (blue) follows them, not y = x.

### r = 0.5

If `x` is 1.5, would you predict `y` = 1.5?

In [ ]:
example <- r_table(0.5)

su_plot(example) +
  geom_vline(xintercept = 1.5, linewidth = 1) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

In [ ]:
nn_prediction_example(1.5)

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_vline(xintercept = 1.5, linewidth = 1) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

What slope do the gold points have? Try 0.5 (blue).

In [ ]:
su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0.5, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

### r = 0.7

Before you run this: what slope will the gold points follow?

In [ ]:
example <- r_table(0.7)
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0.7, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

In standard units, the graph of averages follows the line

$$\text{estimate of } y = r \times x$$

This is the **regression line**.

## Part 3: The Regression Line in Original Units

Back in inches, the regression line has

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}$$

$$\text{intercept} = \text{average of } y - \text{slope} \times \text{average of } x$$

In [ ]:
family_r <- cor(heights$MidParent, heights$Child)
family_r

In [ ]:
family_slope <- family_r * sd(heights$Child) / sd(heights$MidParent)
family_intercept <- mean(heights$Child) - family_slope * mean(heights$MidParent)
c(family_slope, family_intercept)

R fits the same line with `lm()`, short for **linear model**. `Child ~ MidParent` means "predict `Child` from `MidParent`." `coef()` pulls out the intercept and the slope.

In [ ]:
family_model <- lm(Child ~ MidParent, data = heights)
coef(family_model)

Now predict for a midparent height of 74.5 inches. Nearest neighbor couldn't do this one.

In [ ]:
family_slope * 74.5 + family_intercept

Add the regression predictions to the table, next to the nearest neighbor predictions.

In [ ]:
heights_with_predictions <- heights_with_predictions |>
  mutate(`Regression Prediction` = family_slope * MidParent + family_intercept)

heights_with_predictions

In [ ]:
heights_with_predictions |>
  ggplot(aes(x = MidParent)) +
  geom_point(aes(y = Child), alpha = 0.5) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2.5) +
  geom_abline(slope = family_slope, intercept = family_intercept, color = "dodgerblue3", linewidth = 1.2) +
  labs(title = "Nearest neighbor (gold) and regression line (blue)",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

How far apart are the two predictions, on average?

In [ ]:
heights_with_predictions |>
  summarize(average_gap = mean(abs(Prediction - `Regression Prediction`)))

## Try it out!

Predict a penguin's body mass (`body_mass_g`, in grams) from its flipper length (`flipper_length_mm`).

1. Find $r$. In standard units, what slope will the regression line have?
2. Find the slope and intercept with the formula. Check them with `lm()` and `coef()`.
3. Predict the body mass of a penguin with a 220 mm flipper.
4. What do the slope and the intercept mean here?

**Done early?** Use `group_by(species)` to find $r$ within each species. How do they compare with the overall $r$?

In [ ]:
library(palmerpenguins)

# Keep rows with the two variables we need
model_data <- penguins |>
  filter(!is.na(flipper_length_mm), !is.na(body_mass_g)) |>
  select(species, flipper_length_mm, body_mass_g)

model_data

In [ ]:
ggplot(model_data, aes(x = flipper_length_mm, y = body_mass_g, color = species)) +
  geom_point(alpha = 0.8) +
  labs(
    title = "Flipper Length and Body Mass (Palmer Penguins)",
    x = "Flipper Length (mm)",
    y = "Body Mass (g)",
    caption = "Source: palmerpenguins",
    color = "Species"
  ) +
  theme_minimal(base_size = 18)

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Done early? r within each species

## Extra Practice: Hybrid Cars

Last class we looked at hybrid cars. Predict the price, `msrp` (in dollars), from `acceleration` (in km per hour per second).

1. Draw the scatter plot. Does a line look like a good summary?
2. Find $r$. In standard units, what slope will the regression line have?
3. Find the slope and intercept with the formula. Check them with `lm()` and `coef()`.
4. Predict the price of a hybrid with an acceleration of 15.
5. What does the intercept mean here? Does it make sense?

In [ ]:
hybrid <- read_csv("data/hybrid.csv", show_col_types = FALSE)
hybrid

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here